# Análisis de datos - AquaLimpia S. A.

Este notebook presenta un análisis exploratorio del dataset de plantas de tratamiento de aguas residuales de AquaLimpia S. A.

El objetivo es revisar el desempeño de las plantas, calcular la eficiencia de remoción de DBO, analizar el cumplimiento normativo y generar resultados que apoyen la toma de decisiones.


## 1. Importación de librerías

Se importan las librerías necesarias para cargar datos, procesarlos y generar gráficos.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Carga del dataset

Se carga el archivo Excel entregado para el caso de AquaLimpia S. A.

> Nota: este notebook está pensado para estar dentro de la carpeta `notebooks`, por eso la ruta usa `../data/`.


In [ ]:
ruta_datos = Path("../data/dataset_set_A_aguas_residuales.xlsx")

df = pd.read_excel(ruta_datos)

df.head()

## 3. Revisión inicial de los datos

En esta etapa se revisa la cantidad de filas y columnas, los tipos de datos y la existencia de valores nulos.


In [ ]:
print("Cantidad de filas y columnas:")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 4. Preparación de datos

Se transforma la fecha de registro y se calcula la eficiencia de remoción de DBO.

La eficiencia se calcula con la siguiente fórmula:

```text
eficiencia_DBO = ((DBO_entrada - DBO_salida) / DBO_entrada) * 100
```


In [ ]:
df["fecha_registro"] = pd.to_datetime(df["fecha_registro"])

df["eficiencia_DBO"] = (
    (df["DBO_entrada_mg_L"] - df["DBO_salida_mg_L"])
    / df["DBO_entrada_mg_L"]
) * 100

df["estado_cumplimiento"] = np.where(
    df["cumplimiento_norma"] == 1,
    "Cumple",
    "No cumple"
)

df.head()

## 5. Resumen por planta

Se agrupan los datos por planta para comparar el desempeño de cada una.


In [ ]:
resumen_planta = df.groupby("planta").agg(
    total_registros=("planta", "count"),
    caudal_promedio=("caudal_entrada_m3_d", "mean"),
    dbo_entrada_promedio=("DBO_entrada_mg_L", "mean"),
    dbo_salida_promedio=("DBO_salida_mg_L", "mean"),
    eficiencia_promedio=("eficiencia_DBO", "mean"),
    cumplimiento_promedio=("cumplimiento_norma", "mean"),
    energia_promedio=("energia_aeracion_kWh", "mean"),
    lodos_promedio=("lodos_generados_kg_d", "mean")
).reset_index()

resumen_planta["cumplimiento_porcentaje"] = (
    resumen_planta["cumplimiento_promedio"] * 100
)

resumen_planta

## 6. Cumplimiento normativo por planta

Este gráfico permite observar qué plantas presentan mayor o menor cumplimiento de la norma.


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(resumen_planta["planta"], resumen_planta["cumplimiento_porcentaje"])
plt.title("Porcentaje de cumplimiento normativo por planta")
plt.xlabel("Planta")
plt.ylabel("Cumplimiento (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Eficiencia promedio por planta

Este gráfico muestra la eficiencia promedio de remoción de DBO para cada planta.


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(resumen_planta["planta"], resumen_planta["eficiencia_promedio"])
plt.title("Eficiencia promedio de remoción de DBO por planta")
plt.xlabel("Planta")
plt.ylabel("Eficiencia DBO (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Relación entre caudal de entrada y DBO de salida

Este gráfico permite revisar si existe una posible relación entre el caudal recibido por la planta y la DBO de salida.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["caudal_entrada_m3_d"], df["DBO_salida_mg_L"])
plt.title("Relación entre caudal de entrada y DBO de salida")
plt.xlabel("Caudal de entrada m3/d")
plt.ylabel("DBO salida mg/L")
plt.tight_layout()
plt.show()

## 9. Exportación de resultados

Se generan archivos de salida para apoyar la revisión de resultados. Estos archivos quedan en la carpeta `outputs`.


In [ ]:
ruta_outputs = Path("../outputs")
ruta_outputs.mkdir(exist_ok=True)

salida_operaciones = df[
    [
        "fecha_registro",
        "planta",
        "caudal_entrada_m3_d",
        "DBO_entrada_mg_L",
        "DBO_salida_mg_L",
        "energia_aeracion_kWh",
        "lodos_generados_kg_d",
        "eficiencia_DBO"
    ]
]

salida_gestion_ambiental = df[
    [
        "fecha_registro",
        "planta",
        "DBO_salida_mg_L",
        "cumplimiento_norma",
        "estado_cumplimiento"
    ]
]

resumen_planta.to_csv(ruta_outputs / "resumen_por_planta.csv", index=False, encoding="utf-8")
salida_operaciones.to_csv(ruta_outputs / "salida_operaciones.csv", index=False, encoding="utf-8")
salida_gestion_ambiental.to_csv(ruta_outputs / "salida_gestion_ambiental.csv", index=False, encoding="utf-8")

print("Archivos exportados correctamente en la carpeta outputs.")

## 10. Conclusión

El análisis permite comparar el desempeño de las plantas de tratamiento de AquaLimpia S. A. mediante indicadores como la eficiencia de remoción de DBO y el porcentaje de cumplimiento normativo.

Con estos resultados se puede apoyar la toma de decisiones de las áreas de operaciones y gestión ambiental, identificando plantas con más incumplimientos o con menor eficiencia en el proceso.
